In [1]:
from google.colab import drive
drive.mount('/content/drive')
base_dir = "/content/drive/MyDrive/huggingface-rag"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -U sentence-transformers dotenv openai httpx qdrant_client

In [3]:
import os
import httpx
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

qdrant_path = f"{base_dir}/qdrant_db"
collection_name = 'huggingface_transformers_docs'
embedding_model = SentenceTransformer("fyerfyer/finetune-jina-transformers-v1", trust_remote_code=True)

lock_file = os.path.join(qdrant_path, ".lock")
if os.path.exists(lock_file):
  try:
    os.remove(lock_file)
    print(f"Removed stale lock file: {lock_file}")
  except Exception as e:
    print(f"Warning: Could not remove lock file: {e}")


Removed stale lock file: /content/drive/MyDrive/huggingface-rag/qdrant_db/.lock


In [4]:
from google.colab import userdata

class HFRAG:
  def __init__(self):
    self.embed_model = embedding_model
    self.db_client = QdrantClient(path=qdrant_path)
    if not self.db_client.collection_exists(collection_name):
      raise ValueError(f"Cannot find collection {collection_name}, please check qdrant path")
    print(f"Successfully connected to qdrant: {qdrant_path}")

    self.llm_client = OpenAI(
      api_key=userdata.get('DEEPSEEK_API_KEY'),
      base_url="https://api.deepseek.com",
      http_client=httpx.Client(proxy=None, trust_env=False) # 开了代理的话要加这个，不然会报错
    )

  def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.40):
    query_vector = self.embed_model.encode(query).tolist()

    # 在 Colab 中用 Qdrant 的 search 方法好像有些问题
    # 这里弄一个兼容性的
    if hasattr(self.db_client, 'search'):
      results = self.db_client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=top_k,
        score_threshold=score_threshold
      )
    else:
      results = self.db_client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        with_payload=True,
        score_threshold=score_threshold
      ).points

    return results

  def generate(self, query: str, search_results):
    if not search_results:
      return "I'm sorry, but I couldn't find any relevant information in the knowledge base regarding your query."

    context_pieces = []
    for idx, hit in enumerate(search_results, 1):
      source = hit.payload['metadata']['source']
      filename = source.split('/')[-1]
      text = hit.payload['text']

      piece = f"""<doc id="{idx}" source="{filename}">
  {text}
  </doc>"""
      context_pieces.append(piece)

    context_str = "\n\n".join(context_pieces)

    system_prompt = """You are an expert AI assistant specializing in the Hugging Face Transformers library and NLP technology.

  YOUR MISSION:
  Answer the user's question using ONLY the provided "Retrieved Context". Do not rely on your internal knowledge base unless it is to explain syntax or general programming concepts not covered in the documents.

  GUIDELINES:
  1. **Grounding**: Base your answer strictly on the provided context chunks.
  2. **Code First**: If the context contains code examples, prioritize showing them in your answer using Python markdown blocks.
  3. **Citation**: When referencing specific information, cite the source file name (e.g., `[model_doc.md]`).
  4. **Honesty**: If the provided context does not contain enough information to answer the question, state: "The provided documents do not contain the answer to this question." Do not hallucinate or make up parameters.
  5. **Clarity**: Keep explanations concise and technical.

  Output Format:
  - Use Markdown for formatting.
  - Use `code blocks` for function names and parameters.
  """

    # 4. User Prompt (The "Input")
    user_prompt = f"""
  ### User Query
  {query}

  ### Retrieved Context
  Please use the following documents to answer the query above:

  {context_str}

  ### Answer
  """

    print(f"\nThinking (Processing {len(search_results)} context chunks)...")

    try:
      response = self.llm_client.chat.completions.create(
        model="deepseek-chat",
        messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
        max_tokens=4096,
        stream=True
      )

      full_response = ""
      print("-" * 60)
      for chunk in response:
        if chunk.choices[0].delta.content:
          content = chunk.choices[0].delta.content
          print(content, end="", flush=True)
          full_response += content
      print("\n" + "-" * 60)
      return full_response

    except Exception as e:
      return f"Error calling LLM: {e}"

  def chat(self, query: str):
    print(f"\nUser: {query}")
    results = self.retrieve(query)
    self.generate(query, results)

In [ ]:
if __name__ == "__main__":
  rag = HFRAG()

  print("\nHuggingFace RAG assitant is started! Input 'quit' to exit")
  while True:
    user_input = input("\nPlease input your question: ")
    if user_input.lower() in ['quit', 'exit']:
      break

    rag.chat(user_input)

Successfully connected to qdrant: /content/drive/MyDrive/huggingface-rag/qdrant_db

HuggingFace RAG assitant is started! Input 'quit' to exit

Please input your question: How to implement padding?

User: How to implement padding?

Thinking (Processing 5 context chunks)...
------------------------------------------------------------
Based on the provided documents, here's how to implement padding in different contexts:

## Text Tokenization Padding

For text tokenization with Hugging Face tokenizers, use the `padding` parameter:

```python
# Basic padding to longest sequence in batch
encoded_inputs = tokenizer(batch_sentences, padding=True, return_tensors="pt")

# Padding to maximum model input length
encoded_inputs = tokenizer(batch_sentences, padding='max_length', return_tensors="pt")

# Padding to specific length
encoded_inputs = tokenizer(batch_sentences, padding='max_length', max_length=42, return_tensors="pt")

# Padding to multiple of a value
encoded_inputs = tokenizer(batch_sent